# XGBoost SPEI-3 drought forecast

Train an XGBoost regressor to predict each grid cell's SPEI-3 one month ahead. The fitted estimator is saved directly as `model/xgboost_spei3m.pkl`.

In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from xgboost import XGBRegressor

current = Path.cwd().resolve()
ROOT = next((path for path in (current, *current.parents) if (path / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find the project root (pyproject.toml) from the current directory.")

DATA_PATH = ROOT / "data" / "gold" / "features" / "gold_drought_features.parquet"
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Feature dataset not found: {DATA_PATH}")

FEATURES = [
    "temp_mean_celsius", "precip_monthly_mm", "water_balance_mm",
    "soil_moisture_surface", "soil_moisture_rootzone", "spi_3m", "spei_3m",
    "precip_lag_1m", "precip_lag_2m", "spei_3m_lag_1m", "spei_3m_lag_2m",
    "latitude", "longitude", "month_sin", "month_cos",
]
TARGET = "spei_3m"
DROUGHT_THRESHOLD = -1.0

data = pd.read_parquet(DATA_PATH).sort_values(["latitude", "longitude", "valid_time"]).copy()
data["valid_time"] = pd.to_datetime(data["valid_time"])
data["month_sin"] = np.sin(2 * np.pi * data["month"] / 12)
data["month_cos"] = np.cos(2 * np.pi * data["month"] / 12)
grid = data.groupby(["latitude", "longitude"], sort=False)
data["target"] = grid[TARGET].shift(-1)
data["target_time"] = grid["valid_time"].shift(-1)
next_month = data["valid_time"] + pd.offsets.MonthBegin(1)
data = data.loc[data["target_time"].eq(next_month)].dropna(subset=FEATURES + ["target"]).copy()

train = data.loc[data["target_time"] < "2019-01-01"]
validation = data.loc[data["target_time"].between("2019-01-01", "2021-12-31")]
test = data.loc[data["target_time"] >= "2022-01-01"]
if train.empty or validation.empty or test.empty:
    raise ValueError("Time-based train, validation, or test split is empty.")

X_train, y_train = train[FEATURES], train["target"]
X_validation, y_validation = validation[FEATURES], validation["target"]
X_test, y_test = test[FEATURES], test["target"]
print(f"Train: {len(train):,} | validation: {len(validation):,} | test: {len(test):,}")

Train: 72,372 | validation: 11,736 | test: 15,648


In [2]:
model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=1200,
    max_depth=6,
    learning_rate=0.3,
    random_state=42,
    tree_method="hist",
    eval_metric="rmse",
    early_stopping_rounds=50,
)
model.fit(X_train, y_train, eval_set=[(X_validation, y_validation)], verbose=False)

MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / "xgboost_spei3m.pkl"
with MODEL_PATH.open("wb") as model_file:
    pickle.dump(model, model_file, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Best boosting round: {model.best_iteration + 1}")
print(f"Saved fitted model: {MODEL_PATH.relative_to(ROOT)} ({MODEL_PATH.stat().st_size:,} bytes)")

Best boosting round: 17
Saved fitted model: model\xgboost_spei3m.pkl (329,452 bytes)


In [3]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

with MODEL_PATH.open("rb") as model_file:
    saved_model = pickle.load(model_file)
test_predictions = saved_model.predict(X_test)
rmse = float(np.sqrt(mean_squared_error(y_test, test_predictions)))
mae = float(mean_absolute_error(y_test, test_predictions))
r2 = float(r2_score(y_test, test_predictions))
print(f"Test RMSE: {rmse:.4f} | MAE: {mae:.4f} | R2: {r2:.4f}")

Test RMSE: 0.2617 | MAE: 0.2030 | R2: 0.9166
